# NLP Preprocessing Assignment (Production-Quality Submission)
**Course Level:** Third-Year Engineering  
**Topic:** Text Preprocessing Pipeline in NLP


## Task 1: Conceptual Understanding

### 1) Difference between **"Love"** and **"love"** in NLP
In raw text, `"Love"` and `"love"` are treated as different tokens if case normalization is not applied.  
- `"Love"` may appear at sentence start or as a proper noun.  
- `"love"` is usually the regular verb/noun form.  
After lowercasing, both become `"love"`, reducing vocabulary size and improving consistency.

### 2) What happens if stopwords are not removed
If stopwords (like *the, is, and, of*) are not removed:  
- Vocabulary becomes larger and noisier.  
- Important words may get lower weight in simple models (BoW/TF-IDF).  
- Processing becomes slower.  
- Some tasks may still work, but with reduced focus on content words.

### 3) Two real-world scenarios where removing stopwords is harmful
1. **Sentiment Analysis**  
   Words like **"not"** and **"no"** are critical.  
   - `"This is not good"` vs `"This is good"` become similar if `"not"` is removed.

2. **Question Answering / Legal Search**  
   Function words can change intent.  
   - `"right to information"` vs `"right information"`  
   - `"to be or not to be"` loses meaning if stopwords are stripped aggressively.

### 4) Difference between stemming and lemmatization (with examples)
- **Stemming**: cuts word endings using heuristic rules; output may not be a real dictionary word.  
  - `studies -> studi`, `running -> run`
- **Lemmatization**: converts to valid base form (lemma) using vocabulary + grammar.  
  - `studies -> study`, `running -> run`, `better -> good` (with proper POS context)

So, stemming is faster but rough; lemmatization is cleaner and linguistically accurate.


## Task 2: Advanced Preprocessing Function


In [ ]:
import re
from typing import List, Tuple

def preprocess_text(text: str) -> Tuple[List[str], str]:
    """Advanced NLP preprocessing function with edge-case handling."""
    if not isinstance(text, str):
        return [], ""

    text = text.strip()
    if not text:
        return [], ""

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove emails
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' ', text)

    # Remove emojis
    text = re.sub(
        r'[\U0001F300-\U0001F6FF\U0001F700-\U0001F77F\U0001F900-\U0001F9FF'
        r'\U0001FA70-\U0001FAFF\U00002600-\U000027BF]+',
        ' ',
        text
    )

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Lowercase
    text = text.lower()

    # Handle repeated characters (3+ to 1): soooo -> so
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove punctuation
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    if not text:
        return [], ""

    # Tokenize
    raw_tokens = text.split()

    # Remove words with length <= 2, but keep important negations
    keep_short = {"no", "not"}
    tokens = [tok for tok in raw_tokens if len(tok) > 2 or tok in keep_short]

    cleaned_sentence = " ".join(tokens)
    return tokens, cleaned_sentence


## Task 3: Stress Testing (10 Diverse Sentences)


In [ ]:
test_sentences = [
    "I LOOOOVE this movie!!! ???? 10/10",
    "Visit https://example.com NOW!!!",
    "My email is student123@college.edu, ping me.",
    "No way!!! This is not acceptable at all.",
    "bruhhhh this phone is sooo bad lol",
    "COVID19 cases dropped by 2000 in 2023.",
    "HELLO hello HeLLo!!! Are you there??",
    "??????",
    "1234567890",
    "I can't believe it's only Rs. 499!!! What a deaaaal!!!"
]

print("=== Stress Test Results ===\n")
all_tokens = []
processed_tokens_per_sentence = []
clean_sentences = []

for i, sentence in enumerate(test_sentences, 1):
    tokens, cleaned = preprocess_text(sentence)
    processed_tokens_per_sentence.append(tokens)
    clean_sentences.append(cleaned)
    all_tokens.extend(tokens)

    print(f"Sentence {i}")
    print(f"Original Text   : {sentence}")
    print(f"Cleaned Tokens  : {tokens}")
    print(f"Cleaned Sentence: {cleaned}")
    print("-" * 60)


## Task 4: Token Analytics


In [ ]:
def sentence_analytics(tokens: List[str]) -> dict:
    total = len(tokens)
    unique = len(set(tokens))
    avg_len = round(sum(len(t) for t in tokens) / total, 2) if total > 0 else 0.0
    return {
        "total_tokens": total,
        "unique_tokens": unique,
        "avg_token_length": avg_len
    }

print("=== Token Analytics ===\n")
noise_scores = []
meaningful_scores = []

for idx, (original, tokens) in enumerate(zip(test_sentences, processed_tokens_per_sentence), 1):
    stats = sentence_analytics(tokens)
    original_word_count = len(original.split())
    cleaned_word_count = len(tokens)
    removed_ratio = (original_word_count - cleaned_word_count) / max(original_word_count, 1)
    meaningful_score = stats["unique_tokens"]

    noise_scores.append((idx, removed_ratio, original))
    meaningful_scores.append((idx, meaningful_score, original))

    print(f"Sentence {idx}: {stats}")

most_noise = max(noise_scores, key=lambda x: x[1])
most_meaningful = max(meaningful_scores, key=lambda x: x[1])

print("\nMost noise removed:")
print(f"Sentence {most_noise[0]} -> \"{most_noise[2]}\"")
print("\nMost meaningful tokens retained:")
print(f"Sentence {most_meaningful[0]} -> \"{most_meaningful[2]}\"")


## Task 5: Frequency Analysis


In [ ]:
from collections import Counter

word_freq = Counter(all_tokens)
top_10 = word_freq.most_common(10)

if word_freq:
    min_count = min(word_freq.values())
    least_freq_candidates = sorted([w for w, c in word_freq.items() if c == min_count])
    top_5_least = least_freq_candidates[:5]
else:
    top_5_least = []

print("=== Frequency Analysis ===")
print("Top 10 most frequent words:")
for w, c in top_10:
    print(f"{w}: {c}")

print("\nTop 5 least frequent words:")
for w in top_5_least:
    print(f"{w}: {word_freq[w]}")


## Task 6: Full Pipeline


In [ ]:
def full_pipeline(text_list: List[str]) -> dict:
    all_tokens_list = []
    clean_sentences_list = []

    for text in text_list:
        tokens, cleaned = preprocess_text(text)
        all_tokens_list.append(tokens)
        clean_sentences_list.append(cleaned)

    return {
        "tokens": all_tokens_list,
        "clean_sentences": clean_sentences_list
    }

pipeline_output = full_pipeline(test_sentences)
print("=== Full Pipeline Output ===")
print(pipeline_output)


## Task 7: Error Handling


In [ ]:
edge_cases = ["", "??????", "999999"]

print("=== Error Handling Check ===")
for case in edge_cases:
    tokens, cleaned = preprocess_text(case)
    print(f"Input: {repr(case)}")
    print(f"Tokens: {tokens}")
    print(f"Cleaned Sentence: {repr(cleaned)}")
    print("-" * 40)


## Submission Note
This notebook is modular, robust, and executable. It includes conceptual understanding, advanced preprocessing, stress testing, analytics, frequency analysis, full pipeline integration, and edge-case handling in a clean academic format.
